# Historical analysis — public source copy
Outputs and execution counts were cleared for publication privacy. Original evidence remains local.
Do not Run All modeling notebooks, retrain frozen models or reevaluate the consumed test.
Dataset/model assets are withheld; see the release README.

# Phase 5 — Random Forest, fair model comparison and final evaluation

This notebook reads reproducible Phase 5 artifacts. The source modules perform the training; rerunning this notebook **never trains, selects again, or evaluates the held-out test set**. Run the guarded CLI development workflow once before this notebook.

## 1. Business objective and scope
Rank snapshot customers by observed churn label using the existing 18 eligible customer/service fields. This is a retrospective classification exercise, not proof of future churn prediction or intervention effects. No Phase 6/dashboard/deployment is included.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/churn_analytics").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from churn_analytics.phase5_data import read_json
from churn_analytics.modeling import FEATURES
D = ROOT / "reports/phase5"
protocol = read_json(D / "development_protocol.json")
selection = read_json(D / "final_selection_manifest.json")
comparison = pd.read_csv(D / "model_comparison.csv")
assert len(FEATURES) == 18
pd.Series({"dataset_sha256": selection["dataset_sha256"], "split_manifest_id": selection["split_manifest_id"], "training_policy": selection["training_policy"]})

## 2. Established Logistic Regression baseline
The existing artifact and full-precision Phase 4 metrics are preserved. The F01 adapter applies the shared canonical normalization; F02 still strictly validates canonical acquisition. Raw numeric SeniorCitizen must not bypass normalization. Compare identical validation IDs, feature eligibility, target mapping, AP definition and threshold.

In [ ]:
historical = read_json(ROOT / "reports/phase4/metrics.json")
pd.DataFrame({name: values["validation"] for name, values in historical["models"].items()}).loc[["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]]

## 3–6. Trees, ensemble learning, bootstrap sampling and feature randomness
A decision tree recursively partitions customers to improve class purity. Gini impurity is `1 - sum(p_k**2)`; entropy is `-sum(p_k*log2(p_k))`. Information gain is the weighted impurity reduction from a split. Deep trees can overfit small groups.

Random Forest is bagging: train many trees on bootstrap samples (draw training rows with replacement), select random feature subsets at each split, then average tree probabilities. Randomness reduces correlation between trees, often reducing variance. More trees do not remove data bias, missing timestamps, leakage or distribution shift. Depth/leaf limits regularize individual trees.

## 7. Preprocessing and exact split reuse
Training has 4,225 customers; validation and reserved test have 1,409 each. Membership is loaded from the original manifest and checked against dataset fingerprints and stored ID lists; no re-splitting occurs. Development reads only training/validation rows before parsing their non-ID fields.

Both models use the same 18 original features. RF uses train medians and full one-hot encoding, with no numerical scaler. LR retains its original scaler/reference encoding. Median/vocabulary learning occurs inside each CV training fold. Unknown levels are safe inside development encoding; actual scoring uses the strict shared input contract.

In [ ]:
policy = pd.read_csv(ROOT / "reports/phase4/feature_policy.csv")
display(policy)
display(pd.Series(protocol))

## 8. Deterministic Random Forest baselines
Fixed unweighted and balanced_subsample forests use 200 trees, min_samples_leaf=2 and seed 42. One worker bounds local resource use. The implementation is in random_forest.py; the CLI has already run it. Re-executing notebook cells only reads those results.

In [ ]:
audit = read_json(D / "development_audit.json")
pd.DataFrame(audit["candidate_parameters"]).T

## 9–10. Bounded tuning and cross-validation
Eight random configurations, three stratified shuffled folds, seed 42. Hyperparameters vary tree count, depth, split/leaf sizes, feature sampling and class weight. Average Precision is the CV score. SearchCV receives the entire preprocessing Pipeline so each fold independently learns its preprocessing. The best search candidate is refitted on the original training partition only. Fold standard deviation is not a population confidence interval.

In [ ]:
search = read_json(D / "search_audit.json")
display(pd.Series({k: v for k, v in search.items() if k != "cv_membership"}))
cv = pd.read_csv(D / "random_forest_cv_results.csv")
display(cv[["params", "mean_train_score", "mean_test_score", "std_test_score", "rank_test_score"]].sort_values("rank_test_score"))
train_ids = set(audit["train_ids"])
validation_ids = set(audit["validation_ids"])
for fold in search["cv_membership"]:
    assert set(fold["fit_ids"]) | set(fold["score_ids"]) == train_ids
    assert not set(fold["fit_ids"]) & set(fold["score_ids"])
    assert not (set(fold["fit_ids"]) | set(fold["score_ids"])) & validation_ids
print("CV memberships cover only original training IDs; no validation IDs entered CV.")

## 11. Imbalance
Approximately 26.54% of original customers have a churn label. Accuracy can hide poor minority-class recall. Unweighted and balanced_subsample forests are compared explicitly; no SMOTE or fabricated customers are introduced. Class weights alter training emphasis; they do not specify an optimal outreach policy.

## 12–13. Validation metrics and fair comparison
AP summarizes precision-recall ranking; ROC-AUC summarizes true/false positive trade-offs across thresholds. Precision, recall, F1 and accuracy here use 0.50. Every row below describes the same 1,409 validation customers. Historical LR values are checked at full stored precision by the workflow. A point-score difference is not proof of general superiority.

In [ ]:
display(comparison)
for filename in ["validation_model_comparison.png", "validation_roc.png", "validation_precision_recall.png"]:
    display(Image(filename=str(D / "figures" / filename)))

## 14. Threshold trade-offs
The threshold protocol was fixed before RF validation scores: retain 0.50 for all candidates and describe the same seven-threshold grid. Lower thresholds often raise recall and false positives. No costs, outreach capacity, treatment efficacy or benefit outcomes exist here to justify profit optimization. AP/ROC-AUC do not change merely by changing the classification threshold.

In [ ]:
thresholds = pd.read_csv(D / "threshold_analysis.csv")
display(thresholds.loc[thresholds.model_name.eq(selection["selected_model"])])
assert selection["threshold"] == .50
Image(filename=str(D / "figures/validation_threshold_tradeoffs.png"))

## 15. Calibration
A reliable 0.7 prediction would correspond to about 70% observed churn across comparable customers, which is not guaranteed here. Brier score is mean squared probability error; log loss penalizes confidently wrong probabilities. Reliability diagrams use 10 fixed equal-width bins and report bin sizes. No calibrator was fitted, so there is no same-label calibration fitting/evaluation leak. Sparse bins and the snapshot limit interpretation.

In [ ]:
display(comparison[["model_name", "average_precision", "brier_score", "log_loss"]])
display(pd.read_csv(D / "calibration_bins.csv"))
Image(filename=str(D / "figures/validation_calibration.png"))

## 16. Feature importance
Impurity importance aggregates one-hot levels back to original fields. It can favor fields with many split opportunities and share credit among correlated variables. Permutation shuffles each original validation column through the full fitted workflow, using AP, five repeats and seed 42. Repeat SD measures shuffle variability, not a confidence interval. Permutation can create nonphysical service combinations and hide correlated substitutes.

LR coefficients represent signed conditional log-odds contrasts; forest impurity and permutation importance represent different quantities. They are not directly comparable and none proves causality.

In [ ]:
display(pd.read_csv(D / "feature_importance.csv"))
display(pd.read_csv(D / "permutation_importance.csv"))
for filename in ["rf_impurity_importance.png", "rf_permutation_importance.png"]:
    display(Image(filename=str(D / "figures" / filename)))

## 17. Frozen final selection
Highest unrounded validation AP wins; exact ties prefer LR, then fixed RF, balanced RF and tuned RF. No train+validation refit occurs. The manifest binds the chosen artifact hash, threshold, feature contract, training IDs, dataset/split identities, code, protocol and comparison evidence before held-out evaluation.

In [ ]:
pd.Series({key: selection[key] for key in ["selected_model", "artifact_path", "artifact_sha256", "threshold", "frozen_at_utc", "selection_id", "training_policy"]})

## 18. One-time final held-out evaluation
The explicit CLI final-evaluate workflow is authorized only after the frozen manifest and passing guards. It creates a durable claim before reading test labels, performs one prediction call, saves metrics, and marks test consumption. A failed/interrupted claim blocks automatic retries. The original Phase 4 manifest stays historical; Phase 5 test_consumption.json is the authoritative current ledger.

**This cell reads saved metrics only.** Notebook reruns never call final evaluation, never tune from test labels and never refit a model.

In [ ]:
final_path = D / "final_test_metrics.json"
if final_path.exists():
    final = read_json(final_path)
    display(pd.Series(final))
    display(pd.Series(read_json(D / "test_consumption.json")))
else:
    print("Held-out results unavailable. No evaluation is triggered by this notebook.")

## 19–20. Limitations and business conclusions
The sample lacks documented pre-outcome timestamps, a prospective horizon and intervention outcomes. Prior full-data EDA means the holdout was not wholly unseen. Repeated validation reuse, demographic suitability, calibration uncertainty, correlated features and dataset shift limit claims.

Use the findings to propose a prospective outreach study with real costs and treatment response; do not claim retention gains, saved revenue or ROI. Later dashboard work should load the trusted selected artifact through the supported adapter, use the frozen threshold and treat the test as consumed. Read reports/PHASE5_MODEL_COMPARISON_REPORT.md and docs/PHASE5_INTERVIEW_GUIDE.md for actual findings and interview preparation.

References: [Scikit-learn RandomizedSearchCV](https://sklearn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html), [permutation importance](https://sklearn.org/stable/modules/generated/sklearn.inspection.permutation_importance.html).